# PINNs for the viscous Burgers equation: the inverse problem

*PINNs course · Nassim RAZAALY*

In `c_forwardPINNs.ipynb` the equation was **known** and the PINN found the solution $u$. Here we do the opposite: we have **measurements** of $u$, and the equation contains **unknown coefficients**:
$$\partial_t u + \lambda_1\, u\,\partial_x u - \lambda_2\,\partial_{xx} u = 0,\qquad x\in[-1,1],\ t\in[0,0.99].$$
The goal is to find $\lambda_1$ and $\lambda_2$ from the data, together with the solution $u$ everywhere. By default the data are generated with $\lambda_1 = 1$, $\lambda_2 = 0.01/\pi \approx 3.2\cdot10^{-3}$ (the forward problem of `c_forwardPINNs.ipynb`), but any other pair can be chosen.

The idea: $\lambda_1$ and $\lambda_2$ become two **extra trainable parameters**, next to the weights of the network. The loss asks the network to fit the data **and** to satisfy the equation: the only values of $\lambda_1, \lambda_2$ for which both are possible are the true ones.

1. **The data**: measurements of $u$ at random points.
2. **The PINN**: the network, the two unknowns, the loss.
3. **Training**: Adam, then L-BFGS, with live monitoring.
4. **Going further**: two other second-order methods, Levenberg–Marquardt and self-scaled BFGS.
5. **Exercise**: another pair $(\lambda_1, \lambda_2)$.

This is the inverse problem of Raissi *et al.* (J. Comput. Phys., 2019).

In [ ]:
# Download the course library (once): MLP
import os, urllib.request
LIB = "x_lib_training.py"
if not os.path.exists(LIB):
    urllib.request.urlretrieve(f"https://raw.githubusercontent.com/nrazaaly/coursePINNs/main/notebooks/{LIB}", LIB)
from x_lib_training import MLP

import copy, time
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from IPython.display import display
from tqdm.auto import tqdm

torch.set_default_dtype(torch.float32)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE)

## 1. The data

The data are generated with an exact solution. In a real application they would come from measurements or from a simulation.

**Exact solution for any pair $(\lambda_1, \lambda_2)$.** If $u$ solves $\partial_t u + \lambda_1 u\,\partial_x u = \lambda_2\,\partial_{xx}u$, then $v = \lambda_1 u$ solves the standard Burgers equation $\partial_t v + v\,\partial_x v = \lambda_2\,\partial_{xx} v$ (multiply the equation by $\lambda_1$), with viscosity $\lambda_2$ and initial condition $v_0 = -\lambda_1\sin(\pi x)$. The Cole–Hopf formula of `c_forwardPINNs.ipynb` (Section 1.2) still applies: the only change is $f(y) = \exp\big(-\frac{\lambda_1\cos(\pi y)}{2\pi\lambda_2}\big)$, and $\lambda_2$ in place of $\nu$. The solution keeps $u(x,0) = -\sin(\pi x)$ and $u(\pm1,t) = 0$; its front has a width of the order of $\lambda_2/\lambda_1$.

In [ ]:
LAMBDA1_TRUE, LAMBDA2_TRUE = 1.0, 0.01 / np.pi        # true coefficients used to generate the data (unknown for the PINN)
T_MAX = 0.99


def burgers_exact(x, t, lambda1=LAMBDA1_TRUE, lambda2=LAMBDA2_TRUE, nz=2001, L=12.0, chunk=1000):
    """Exact u(x, t) of u_t + lambda1 u u_x = lambda2 u_xx, u(x,0) = -sin(pi x), by Cole-Hopf. x, t: arrays of the same shape."""
    x, t = np.broadcast_arrays(np.asarray(x, dtype=float), np.asarray(t, dtype=float))
    shape, x, t = x.shape, x.ravel(), t.ravel()
    z = np.linspace(-L, L, nz)
    u = np.empty(x.size)
    for i in range(0, x.size, chunk):
        xi, ti = x[i:i + chunk, None], t[i:i + chunk, None]
        y = xi - np.sqrt(4 * lambda2 * ti) * z
        E = -lambda1 * np.cos(np.pi * y) / (2 * np.pi * lambda2) - z ** 2
        w = np.exp(E - E.max(axis=1, keepdims=True))
        u[i:i + chunk] = -(np.sin(np.pi * y) * w).sum(1) / w.sum(1)
    return u.reshape(shape)


# reference grid (256 x 100), only used to evaluate the PINN at the end
X_REF, T_REF = np.linspace(-1, 1, 256), np.linspace(0, T_MAX, 100)
U_REF = burgers_exact(X_REF[:, None], T_REF[None, :])
XT_REF = np.stack(np.meshgrid(X_REF, T_REF, indexing="ij"), axis=-1).reshape(-1, 2)

Two sets of points:

* **data points**: $N_d = 2000$ random points $(x_i, t_i)$ where $u$ is measured;
* **residual points**: $N_r = 5000$ random points where the equation is checked (no measurement needed there).

There are **no** initial or boundary conditions: the data replace them.

In [ ]:
N_DATA, N_RES = 2000, 5000
rng = np.random.default_rng(0)
xt = np.c_[rng.uniform(-1, 1, N_DATA), rng.uniform(0, T_MAX, N_DATA)]
xt_data = torch.tensor(xt, dtype=torch.float32, device=DEVICE)                                  # (N_DATA, 2)
u_data = torch.tensor(burgers_exact(xt[:, 0], xt[:, 1])[:, None], dtype=torch.float32, device=DEVICE)   # (N_DATA, 1)
xt_res = torch.tensor(np.c_[rng.uniform(-1, 1, N_RES), rng.uniform(0, T_MAX, N_RES)], dtype=torch.float32, device=DEVICE)

plt.figure(figsize=(11, 4))
plt.imshow(U_REF, extent=[0, T_MAX, -1, 1], origin="lower", aspect="auto", cmap="coolwarm", vmin=-1, vmax=1, alpha=0.6)
plt.colorbar(label="exact $u$")
plt.scatter(xt_res[:, 1].cpu(), xt_res[:, 0].cpu(), s=0.5, c="gray", label=f"{N_RES} residual points")
plt.scatter(xt_data[:, 1].cpu(), xt_data[:, 0].cpu(), s=4, c="k", label=f"{N_DATA} data points")
plt.xlabel("$t$"); plt.ylabel("$x$"); plt.legend(loc="upper left", bbox_to_anchor=(1.15, 1)); plt.tight_layout(); plt.show()

## 2. The PINN

### 2.1 The network

The network of Raissi *et al.*: 8 hidden layers of 20 neurons, `tanh`, input $(x, t)$, output $u$.

In [ ]:
torch.manual_seed(0)
model = MLP([2, 20, 20, 20, 20, 20, 20, 20, 20, 1], activation=nn.Tanh).to(DEVICE)
print(model)
model.show()

### 2.2 The two unknowns

$\lambda_1$ and $\lambda_2$ are two `nn.Parameter`: the optimizer updates them like the weights of the network.

* $\lambda_1$ is trained directly, starting from $0$.
* $\lambda_2$ is trained through its logarithm, $\lambda_2 = e^{\log\lambda_2}$, starting from $\log\lambda_2 = -6$ ($\lambda_2 \approx 2.5\cdot10^{-3}$). Two reasons: $\lambda_2$ stays **positive** (a negative viscosity would be an anti-diffusion), and the optimizer moves $\log\lambda_2$ by steps of order $10^{-3}$, i.e. **relative** changes of $\lambda_2$, as for $\lambda_1 \approx 1$. Trained directly, a step of $10^{-3}$ would be 30 % of $\lambda_2$.

In [ ]:
lambda1 = nn.Parameter(torch.tensor(0.0, device=DEVICE))
log_lambda2 = nn.Parameter(torch.tensor(-6.0, device=DEVICE))
print(f"initial guess: lambda1 = {lambda1.item():.3f}, lambda2 = {torch.exp(log_lambda2).item():.2e} | true values: {LAMBDA1_TRUE}, {LAMBDA2_TRUE:.2e}")

### 2.3 The loss

$$\mathcal L = \underbrace{\frac{1}{N_d}\sum_{i=1}^{N_d}\big(u_\theta(x_i,t_i) - u_i\big)^2}_{\text{data}} + \underbrace{\frac{1}{N_r}\sum_{j=1}^{N_r} r(x_j,t_j)^2}_{\text{PDE residual}},\qquad r = \partial_t u_\theta + \lambda_1\,u_\theta\,\partial_x u_\theta - \lambda_2\,\partial_{xx} u_\theta.$$

As for the forward problem, with a **data** term instead of the initial and boundary terms, and $\lambda_1, \lambda_2$ instead of $1, \nu$ in the residual. The derivatives are computed as in `c_forwardPINNs.ipynb` (Section 2.2).

In [ ]:
def derivatives(model, xt):
    """u, u_x, u_t, u_xx at the points xt (N, 2), by automatic differentiation. model: (N,2)-> (N,1)"""
    xt = xt.clone().requires_grad_(True)
    u = model(xt)
    du = torch.autograd.grad(u, xt, torch.ones_like(u), create_graph=True)[0]       # (N, 2) = (u_x, u_t)
    u_x, u_t = du[:, 0:1], du[:, 1:2]
    u_xx = torch.autograd.grad(u_x, xt, torch.ones_like(u_x), create_graph=True)[0][:, 0:1]
    return u, u_x, u_t, u_xx


u, u_x, u_t, u_xx = derivatives(model, xt_res)
r = u_t + lambda1 * u * u_x - torch.exp(log_lambda2) * u_xx
print(f"initial loss: data {((model(xt_data) - u_data) ** 2).mean().item():.2e} | PDE {(r ** 2).mean().item():.2e}")

## 3. Training: Adam, then L-BFGS

We use two optimizers, one after the other, on the full set of points (no mini-batches): **Adam**, for `N_ADAM` iterations, then **L-BFGS**, a quasi-Newton method, for `N_LBFGS` iterations, starting from the Adam solution. Both update **all** the unknowns: the weights of the network, $\lambda_1$ and $\log\lambda_2$.

All the choices of the training are made in the **initialization cell** below: the network, the two unknowns, the **weights of the loss** and the two optimizers. Run it again before any new training (otherwise the training continues from the current state).

**The loss, with weights.** The loss of Section 2.3 is written explicitly in the training loops, with one weight per term:
$$\mathcal L(\theta, \lambda_1, \lambda_2) = w_{\mathrm{data}}\,\mathrm{MSE}_{\mathrm{data}} + w_{\mathrm{PDE}}\,\mathrm{MSE}_{\mathrm{PDE}}.$$
The two weights `W_DATA`, `W_PDE` are set to 1 (the loss of Section 2.3).

**Adam with a cosine schedule.** The learning rate decreases from $10^{-2}$ to $10^{-7}$ at the last iteration: large steps first, then small ones, so that Adam ends close to a minimum, a good starting point for L-BFGS.

**Monitoring.** Every `LOG_EVERY = 100` iterations we store in `hist` the loss and its two terms (before weighting), $\lambda_1$ and $\lambda_2$, and we redraw the live plots.

In [ ]:
# ---- Initialization: model, unknowns, loss weights, optimizers  (run this cell again before a new training)
from torch.nn.utils import parameters_to_vector     # all the parameters in one vector (stop test of L-BFGS)

# Model (as in Section 2.1) and the two unknowns (as in Section 2.2)
torch.manual_seed(0)
model = MLP([2, 20, 20, 20, 20, 20, 20, 20, 20, 1], activation=nn.Tanh).to(DEVICE)
lambda1 = nn.Parameter(torch.tensor(0.0, device=DEVICE))
log_lambda2 = nn.Parameter(torch.tensor(-6.0, device=DEVICE))
params = list(model.parameters()) + [lambda1, log_lambda2]          # all the unknowns

# Weights of the loss: loss = W_DATA * MSE_data + W_PDE * MSE_PDE
W_DATA = 1.0                                        # data misfit
W_PDE = 1.0                                         # PDE residual

# Optimizers
N_ADAM, N_LBFGS = 2000, 4000                        # iterations of each optimizer
LOG_EVERY = 100                                     # monitoring and live plot every LOG_EVERY iterations
optimizer_adam = torch.optim.Adam(params, lr=1e-2)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_adam, T_max=N_ADAM, eta_min=1e-7)   # lr: 1e-2 -> 1e-7
optimizer_lbfgs = torch.optim.LBFGS(params, lr=1.0,
                                    max_iter=LOG_EVERY, max_eval=10 * LOG_EVERY,     # per call to .step()
                                    history_size=50,
                                    tolerance_grad=0.0, tolerance_change=0.0,        # no convergence test
                                    line_search_fn="strong_wolfe")

# History of the training (one value every LOG_EVERY iterations)
hist = {"it": [], "time": [], "loss": [], "data": [], "pde": [], "lambda1": [], "lambda2": [], "lr": []}
print(model)

### 3.1 Adam

The standard loop: at each iteration, the two terms of the loss are computed on all the points, then their weighted sum; its gradient is back-propagated, all the unknowns are updated, then the learning rate. Every `LOG_EVERY` iterations, the values are stored in `hist` and the plots are redrawn: loss, learning rate, $\lambda_1$ and $\lambda_2$.

In [ ]:
# ---- Training with Adam
t0 = time.time()
fig, axs = plt.subplot_mosaic([["loss", "l1"], ["lr", "l2"]], figsize=(13, 6), layout="constrained")
live = display(fig, display_id=True); plt.close(fig)

for it in tqdm(range(N_ADAM), desc="Adam"):
    # 1. the loss: its two terms, then their weighted sum
    optimizer_adam.zero_grad()
    u, u_x, u_t, u_xx = derivatives(model, xt_res)
    r = u_t + lambda1 * u * u_x - torch.exp(log_lambda2) * u_xx     # PDE residual at the residual points
    loss_pde = (r ** 2).mean()
    loss_data = ((model(xt_data) - u_data) ** 2).mean()            # misfit at the data points
    loss = W_DATA * loss_data + W_PDE * loss_pde

    # 2. gradient, update of all the unknowns, then of the learning rate
    loss.backward()
    optimizer_adam.step()
    scheduler.step()

    # 3. monitoring and live plot
    if it % LOG_EVERY == 0 or it == N_ADAM - 1:
        hist["it"].append(it)
        hist["time"].append(time.time() - t0)
        hist["loss"].append(loss.item())
        hist["data"].append(loss_data.item())
        hist["pde"].append(loss_pde.item())
        hist["lambda1"].append(lambda1.item())
        hist["lambda2"].append(torch.exp(log_lambda2).item())
        hist["lr"].append(scheduler.get_last_lr()[0])

        for ax in axs.values(): ax.clear()
        axs["loss"].semilogy(hist["it"], hist["loss"], color="tab:blue", lw=2.2, alpha=0.5, label="total loss")
        axs["loss"].semilogy(hist["it"], hist["data"], color="tab:green", label="data")
        axs["loss"].semilogy(hist["it"], hist["pde"], color="tab:orange", label="PDE")
        axs["lr"].semilogy(hist["it"], hist["lr"])
        axs["l1"].plot(hist["it"], hist["lambda1"]); axs["l1"].axhline(LAMBDA1_TRUE, color="k", ls="--")
        axs["l2"].plot(hist["it"], hist["lambda2"]); axs["l2"].axhline(LAMBDA2_TRUE, color="k", ls="--")
        axs["loss"].legend(); axs["loss"].set_title(f"loss = {W_DATA:g} data + {W_PDE:g} PDE")
        axs["lr"].set_title("learning rate (cosine schedule)")
        axs["l1"].set_title(r"$\lambda_1$ (dashed: true)"); axs["l2"].set_title(r"$\lambda_2$ (dashed: true)")
        for ax in axs.values(): ax.set_xlabel("iteration")
        live.update(fig)

time_adam = time.time() - t0
print(f"after Adam  : {N_ADAM} iterations, {time_adam:.0f} s | loss {hist['loss'][-1]:.2e} "
      f"(data {hist['data'][-1]:.1e}, PDE {hist['pde'][-1]:.1e}) | lambda1 = {lambda1.item():.4f}, lambda2 = {torch.exp(log_lambda2).item():.3e}")
adam_state = (copy.deepcopy(model.state_dict()), lambda1.item(), log_lambda2.item())   # saved for Section 4

**Analysis.**
* The data term decreases fast: the network first fits the measurements.
* The PDE term jumps from $10^{-5}$ (the initial network is almost constant, its derivatives are small) to $10^{-3}$: Adam cannot fit the data and satisfy the equation at the same time while $\lambda_1, \lambda_2$ are wrong. While the learning rate is large, the training has some jumps (around iteration 800); with the cosine schedule, the end of the training is smooth.
* $\lambda_1$ and $\lambda_2$ move towards the true values, but slowly: after 2000 iterations $\lambda_1 \approx 0.8$ (20 % error), and $\lambda_2$ is about 20 % too large.

Adam is robust from a random start, but slow to converge: a second-order method takes over.

### 3.2 L-BFGS

An L-BFGS iteration computes a direction $\mathbf p_k = -H_k\nabla\mathcal L(\theta_k)$ ($H_k$: approximation of the inverse Hessian, built from the last 50 steps and changes of gradient), then a step length $\eta$ along it, $\theta_{k+1} = \theta_k + \eta\,\mathbf p_k$. With `line_search_fn="strong_wolfe"`, $\eta$ must satisfy the two **strong Wolfe conditions**:
$$\underbrace{\mathcal L(\theta_k + \eta\,\mathbf p_k) \le \mathcal L(\theta_k) + c_1\,\eta\,\nabla\mathcal L(\theta_k)^\top\mathbf p_k}_{\text{sufficient decrease}},\qquad \underbrace{\big|\nabla\mathcal L(\theta_k + \eta\,\mathbf p_k)^\top\mathbf p_k\big| \le c_2\,\big|\nabla\mathcal L(\theta_k)^\top\mathbf p_k\big|}_{\text{the slope has decreased enough}},\qquad c_1 = 10^{-4},\ c_2 = 0.9.$$
To check them, the line search needs the loss **and** its gradient at each trial point $\theta_k + \eta\,\mathbf p_k$: it tries $\eta = 1$, then smaller or larger values, until both conditions hold. So the loss must be computed **several times** per iteration, at points chosen by the optimizer: we give it a function, the `closure`, which computes the loss (the same weighted loss as for Adam) and its gradient at the current parameters, and that `optimizer_lbfgs.step(closure)` calls as many times as needed.

Each call to `.step(closure)` performs `max_iter = LOG_EVERY` iterations; between two calls, L-BFGS keeps its memory, so `N_LBFGS // LOG_EVERY` calls make `N_LBFGS` iterations. After each call, we record and redraw the plots, with two more panels: the relative errors on $\lambda_1$ and $\lambda_2$ (log scale).

| L-BFGS argument | value | meaning |
|---|---|---|
| `lr` | 1 | first trial step $\eta=1$, then adjusted by the line search |
| `line_search_fn` | `"strong_wolfe"` | the step must satisfy the two Wolfe conditions |
| `history_size` | 50 | number of pairs (step, change of gradient) kept to build $H_k$ |
| `max_iter`, `max_eval` | 100, 1000 | iterations and loss evaluations per call to `.step()`; we call it repeatedly |
| `tolerance_grad`, `tolerance_change` | 0 | no convergence test: we decide the number of iterations |

The loop stops earlier if a call to `.step()` does not move the parameters: the line search found no decrease (typically in float32, when the changes of the loss reach the rounding level).

In [ ]:
# ---- Training with L-BFGS (starts from the state after Adam)
def closure():
    # the loss: its two terms, then their weighted sum (as for Adam)
    optimizer_lbfgs.zero_grad()
    u, u_x, u_t, u_xx = derivatives(model, xt_res)
    r = u_t + lambda1 * u * u_x - torch.exp(log_lambda2) * u_xx
    loss_pde = (r ** 2).mean()
    loss_data = ((model(xt_data) - u_data) ** 2).mean()
    loss = W_DATA * loss_data + W_PDE * loss_pde
    loss.backward()
    terms.update(data=loss_data.item(), pde=loss_pde.item())       # kept for the monitoring
    return loss


terms = {}                                                         # the two terms at the last call of closure()
t0 = time.time()
fig, axs = plt.subplot_mosaic([["loss", "l1", "e1"], ["loss", "l2", "e2"]], figsize=(17, 6), layout="constrained")
live = display(fig, display_id=True); plt.close(fig)

for call in tqdm(range(N_LBFGS // LOG_EVERY), desc="L-BFGS"):
    # 1. LOG_EVERY iterations of L-BFGS
    theta_old = parameters_to_vector(params).clone()
    optimizer_lbfgs.step(closure)

    # 2. monitoring (at the current parameters) and live plot
    loss = closure()
    n_iter = optimizer_lbfgs.state[optimizer_lbfgs.param_groups[0]["params"][0]]["n_iter"]   # L-BFGS iterations done
    hist["it"].append(N_ADAM + n_iter)
    hist["time"].append(time_adam + time.time() - t0)
    hist["loss"].append(loss.item())
    hist["data"].append(terms["data"])
    hist["pde"].append(terms["pde"])
    hist["lambda1"].append(lambda1.item())
    hist["lambda2"].append(torch.exp(log_lambda2).item())

    l1, l2 = np.array(hist["lambda1"]), np.array(hist["lambda2"])
    for ax in axs.values(): ax.clear()
    axs["loss"].semilogy(hist["it"], hist["loss"], color="tab:blue", lw=2.2, alpha=0.5, label="total loss")
    axs["loss"].semilogy(hist["it"], hist["data"], color="tab:green", label="data")
    axs["loss"].semilogy(hist["it"], hist["pde"], color="tab:orange", label="PDE")
    axs["l1"].plot(hist["it"], l1); axs["l1"].axhline(LAMBDA1_TRUE, color="k", ls="--")
    axs["l2"].plot(hist["it"], l2); axs["l2"].axhline(LAMBDA2_TRUE, color="k", ls="--")
    axs["e1"].semilogy(hist["it"], np.abs(l1 - LAMBDA1_TRUE) / LAMBDA1_TRUE)
    axs["e2"].semilogy(hist["it"], np.abs(l2 - LAMBDA2_TRUE) / LAMBDA2_TRUE)
    axs["loss"].legend(); axs["loss"].set_title(f"loss = {W_DATA:g} data + {W_PDE:g} PDE")
    axs["l1"].set_title(r"$\lambda_1$ (dashed: true)"); axs["l2"].set_title(r"$\lambda_2$ (dashed: true)")
    axs["e1"].set_title(r"relative error on $\lambda_1$"); axs["e2"].set_title(r"relative error on $\lambda_2$")
    for ax in axs.values():
        ax.axvline(N_ADAM, color="k", ls=":", lw=1)                # dotted line: Adam -> L-BFGS
        ax.set_xlabel("iteration")
    live.update(fig)

    # 3. stop if the parameters did not move (the line search found no decrease)
    if torch.equal(theta_old, parameters_to_vector(params)):
        print(f"L-BFGS: no decrease found by the line search, stop after {n_iter} iterations")
        break

print(f"after L-BFGS: {hist['it'][-1]} iterations, {hist['time'][-1]:.0f} s | loss {hist['loss'][-1]:.2e} "
      f"(data {hist['data'][-1]:.1e}, PDE {hist['pde'][-1]:.1e}) | lambda1 = {lambda1.item():.5f}, lambda2 = {torch.exp(log_lambda2).item():.4e}")

### 3.3 Results

In [ ]:
l1, l2 = lambda1.item(), torch.exp(log_lambda2).item()
with torch.no_grad():
    U = model(torch.tensor(XT_REF, dtype=lambda1.dtype, device=DEVICE)).cpu().numpy().reshape(U_REF.shape)
print(f"lambda1 = {l1:.5f}    (true {LAMBDA1_TRUE:.5f})    -> error {100 * abs(l1 - LAMBDA1_TRUE) / LAMBDA1_TRUE:.2f} %")
print(f"lambda2 = {l2:.4e} (true {LAMBDA2_TRUE:.4e}) -> error {100 * abs(l2 - LAMBDA2_TRUE) / LAMBDA2_TRUE:.2f} %")
print(f"relative L2 error on u: {np.linalg.norm(U - U_REF) / np.linalg.norm(U_REF):.1e}")

fig, axs = plt.subplots(1, 4, figsize=(17, 3.6))
im = axs[0].imshow(np.abs(U - U_REF), extent=[0, T_MAX, -1, 1], origin="lower", aspect="auto", cmap="magma_r")
fig.colorbar(im, ax=axs[0]); axs[0].set(title=r"$|u_\theta - u|$", xlabel="$t$", ylabel="$x$")
for ax, k in zip(axs[1:], [25, 50, 75]):
    ax.plot(X_REF, U_REF[:, k], "k", lw=3, alpha=0.3, label="exact")
    ax.plot(X_REF, U[:, k], "r--", lw=1.5, label="PINN")
    ax.set(title=f"$t = {T_REF[k]:.2f}$", xlabel="$x$")
axs[1].legend(); plt.tight_layout(); plt.show()

**Analysis.**
* L-BFGS brings $\lambda_1$ and $\lambda_2$ to their true values within a few hundred iterations; both loss terms decrease by 3 to 4 orders of magnitude.
* $\lambda_1$ is found with an error of about $10^{-4}$ to $10^{-3}$, $\lambda_2$ with an error of about $10^{-3}$. The solution $u$ is reconstructed everywhere, with a relative error below $10^{-3}$; the largest errors are near $x = 0$, in the front.

**Why is $\lambda_2$ harder?** The term $\lambda_2\,\partial_{xx}u$ is negligible everywhere, except in the thin front ($|x| \lesssim 10^{-2}$), where $\partial_{xx} u$ is huge. So $\lambda_2$ is only "seen" by the few data and residual points that lie in the front (a few out of 2000): a small change of $\lambda_2$ hardly changes the loss. To find it more precisely, two levers:
* **minimize the loss much better**: L-BFGS in float32 stalls around a loss of $10^{-6}$; a better second-order optimizer, in float64, goes much lower and reaches an error of a few $10^{-5}$ on $\lambda_2$ (Section 4);
* **more data**, in particular in the front (exercise 4 of Section 5).

## 4. Going further: other second-order methods

Two methods, written as functions. Each one performs **one iteration** on all the unknowns `params`; in the training loop (Section 4.3), you choose the method.

### 4.1 Levenberg–Marquardt

The loss is a **sum of squares**: $\mathcal L(\theta) = \|f(\theta)\|^2$, where $f \in \mathbb R^m$ gathers the $m = N_r + N_d$ residuals, $\sqrt{w_{\rm PDE}/N_r}\,r_j$ and $\sqrt{w_{\rm data}/N_d}\,(u_\theta(x_i,t_i) - u_i)$, and $\theta \in \mathbb R^n$ the $n$ unknowns (weights, $\lambda_1$, $\log\lambda_2$). With the Jacobian $J = \partial f/\partial\theta$ ($m\times n$), the **Gauss–Newton** method linearizes $f(\theta + d) \approx f + Jd$ and minimizes $\|f + Jd\|^2$; **Levenberg–Marquardt** adds a damping $\mu$:
$$(J^\top J + \mu I)\,d = -J^\top f,\qquad \theta \leftarrow \theta + d.$$
Near a minimum, $J^\top J$ approximates the Hessian of $\mathcal L$ (up to a factor 2) very well, so each iteration is worth many L-BFGS iterations. The price: the Jacobian and an $n\times n$ linear system at each iteration, affordable here ($n = 3023$).

**Damping.** We compare the actual decrease of the loss with the decrease predicted by the linear model, $\rho = \frac{\mathcal L(\theta) - \mathcal L(\theta + d)}{\|f\|^2 - \|f + Jd\|^2}$: if $\rho > 0.75$, $\mu$ is divided by 3 (larger steps); if $\rho < 0.25$, $\mu$ is doubled; if $\rho \le 0$, the step is rejected and computed again with the larger $\mu$.

This is the **Gauss–Newton natural gradient** of Jnini, Vella and Zeinhofer (2024) for PINNs; their *dual* version (2025) solves the system in the space of the residuals ($m\times m$) when $m < n$.

**The Jacobian, with `torch.func`.** We write the residual at **one** point as a function of the parameters (`functional_call(model, p, x)` evaluates the network with the parameters `p`, a dictionary), differentiate it with `jacrev`, and vectorize over the points with `vmap`.

In [ ]:
from torch.func import functional_call, grad, jacrev, vmap


def u_point(p, xt):
    """u at one point xt = (x, t), with the network parameters p (dictionary)"""
    return functional_call(model, p, (xt.unsqueeze(0),))[0, 0]


def residual_point(p, l1, ll2, xt):
    """PDE residual at one point"""
    u = u_point(p, xt)
    u_x, u_t = grad(u_point, argnums=1)(p, xt)
    u_xx = grad(lambda z: grad(u_point, argnums=1)(p, z)[0])(xt)[0]
    return u_t + l1 * u * u_x - torch.exp(ll2) * u_xx


def data_point(p, l1, ll2, xt):
    """network output at one data point (l1, ll2 unused)"""
    return u_point(p, xt)


def residuals(jacobian=False):
    """f (m,) such that loss = sum(f^2), and its Jacobian J (m, n) with respect to (weights, lambda1, log_lambda2)"""
    p = {k: v.detach() for k, v in model.named_parameters()}
    args = (p, lambda1.detach(), log_lambda2.detach())
    f = torch.cat([vmap(residual_point, in_dims=(None, None, None, 0))(*args, xt_res) * (W_PDE / N_RES) ** 0.5,
                   (vmap(data_point, in_dims=(None, None, None, 0))(*args, xt_data) - u_data[:, 0]) * (W_DATA / N_DATA) ** 0.5])
    if not jacobian:
        return f
    J = []
    for fun, xt, w in [(residual_point, xt_res, W_PDE / N_RES), (data_point, xt_data, W_DATA / N_DATA)]:
        Jp, J1, J2 = vmap(jacrev(fun, argnums=(0, 1, 2)), in_dims=(None, None, None, 0), chunk_size=500)(*args, xt)
        J.append(torch.cat([Jp[k].reshape(len(xt), -1) for k in p] + [J1[:, None], J2[:, None]], dim=1) * w ** 0.5)
    return f, torch.cat(J)


def levenberg_marquardt_step(params, state):
    """One Levenberg-Marquardt iteration. state: dictionary with the damping 'mu' (updated)."""
    f, J = residuals(jacobian=True)
    loss = (f ** 2).sum().item()
    theta = nn.utils.parameters_to_vector(params).detach()
    I = torch.eye(len(theta), dtype=theta.dtype, device=theta.device)
    while True:
        d = torch.linalg.solve(J.T @ J + state["mu"] * I, -J.T @ f)
        predicted = loss - ((f + J @ d) ** 2).sum().item()        # decrease predicted by the linear model
        nn.utils.vector_to_parameters(theta + d, params)
        rho = (loss - (residuals() ** 2).sum().item()) / predicted
        if rho > 0.75: state["mu"] /= 3
        if rho < 0.25: state["mu"] *= 2
        if rho > 0: return
        nn.utils.vector_to_parameters(theta, params)                # step rejected

### 4.2 Self-scaled BFGS

L-BFGS approximates the inverse Hessian $H \approx (\nabla^2\mathcal L)^{-1}$ with the last 50 pairs (step, change of gradient). With $n = 3023$ unknowns, the **full** $n\times n$ matrix fits in memory (73 MB in float64): this is **BFGS**. Recent PINN studies (Urbán *et al.*, J. Comput. Phys. 2025; Kiyani *et al.*, 2025) show that its **self-scaled** version, by Al-Baali, reaches errors several orders of magnitude below L-BFGS. At each iteration:
$$\mathbf p = -H\nabla\mathcal L(\theta),\qquad \theta \leftarrow \theta + \eta\,\mathbf p\ \ (\eta \text{ from a strong Wolfe line search}),\qquad \mathbf s = \eta\,\mathbf p,\quad \mathbf y = \nabla\mathcal L(\theta_{\rm new}) - \nabla\mathcal L(\theta),$$
$$H \leftarrow \max(1, b)\,H,\quad b = \frac{-\eta\,\mathbf s^\top\nabla\mathcal L(\theta)}{\mathbf s^\top\mathbf y},\qquad H \leftarrow \big(I - \rho\,\mathbf s\mathbf y^\top\big)\,H\,\big(I - \rho\,\mathbf y\mathbf s^\top\big) + \rho\,\mathbf s\mathbf s^\top,\quad \rho = \frac{1}{\mathbf s^\top\mathbf y}.$$
The second line is the BFGS update, after a scaling of $H$: $b > 1$ means that $H$ is too small along $\mathbf s$. At the first iteration, $H = I$, then $H = \frac{\mathbf s^\top\mathbf y}{\mathbf y^\top\mathbf y}I$ before the update. The update is skipped if $\mathbf s^\top\mathbf y \le 0$. The line search is the one of `torch.optim.LBFGS` (`_strong_wolfe`, a private function of PyTorch).

In [ ]:
from torch.optim.lbfgs import _strong_wolfe


def loss_and_grad(params, theta):
    """total loss and its gradient (flattened) at the parameters theta (flattened)"""
    nn.utils.vector_to_parameters(theta, params)
    for p in params:
        p.grad = None
    u, u_x, u_t, u_xx = derivatives(model, xt_res)
    r = u_t + lambda1 * u * u_x - torch.exp(log_lambda2) * u_xx
    loss = W_DATA * ((model(xt_data) - u_data) ** 2).mean() + W_PDE * (r ** 2).mean()
    loss.backward()
    return loss.item(), torch.cat([p.grad.reshape(-1) for p in params])


def ssbfgs_step(params, state):
    """One self-scaled BFGS iteration. state: dictionary with H, the loss and the gradient (updated)."""
    theta = nn.utils.parameters_to_vector(params).detach()
    first = "H" not in state
    if first:
        state["loss"], state["g"] = loss_and_grad(params, theta)
        state["H"] = torch.eye(len(theta), dtype=theta.dtype, device=theta.device)
    H, g = state["H"], state["g"]
    p = -H @ g                                                        # search direction
    t = min(1.0, 1.0 / g.abs().sum().item()) if first else 1.0       # first trial step (as in L-BFGS)
    obj = lambda x, t, d: loss_and_grad(params, x + t * d)
    loss, g_new, eta, _ = _strong_wolfe(obj, theta, t, p, state["loss"], g, g @ p)
    s, y = eta * p, g_new - g
    nn.utils.vector_to_parameters(theta + s, params)
    sy = (s @ y).item()
    if sy > 0:
        if first:
            H = sy / (y @ y).item() * torch.eye(len(theta), dtype=theta.dtype, device=theta.device)
        else:
            H = max(1.0, -eta * (s @ g).item() / sy) * H                # self-scaling (Al-Baali)
        rho = 1 / sy                                                  # BFGS update, expanded: no n x n products
        Hy = H @ y
        H = H - rho * (torch.outer(s, Hy) + torch.outer(Hy, s)) + (rho ** 2 * (y @ Hy).item() + rho) * torch.outer(s, s)
    state.update(H=H, g=g_new, loss=loss)

### 4.3 Training at second order

As for L-BFGS: (1) the model and the optimizer, (2) the training loop, (3) the check. The starting point is the same: the network and $\lambda_1, \lambda_2$ after Adam (Section 3.1). Both methods work in float64: near the minimum, they use differences of very close numbers.

**(1) Model and optimizer.** Choose the method with `METHOD`. `state` is the memory of the method between two iterations: the damping $\mu$ for Levenberg–Marquardt, the matrix $H$, the loss and the gradient for self-scaled BFGS.

In [ ]:
METHOD = "SSBFGS"                                                   # "SSBFGS" (self-scaled BFGS) or "LM" (Levenberg-Marquardt)
step = {"SSBFGS": ssbfgs_step, "LM": levenberg_marquardt_step}[METHOD]
state = {"mu": 1e-3}

model = MLP([2, 20, 20, 20, 20, 20, 20, 20, 20, 1], activation=nn.Tanh).to(DEVICE)
model.load_state_dict(adam_state[0])                                # the network after Adam
model.double()
print(model)
model.show()
lambda1 = nn.Parameter(torch.tensor(adam_state[1], dtype=torch.float64, device=DEVICE))
log_lambda2 = nn.Parameter(torch.tensor(adam_state[2], dtype=torch.float64, device=DEVICE))
params = list(model.parameters()) + [lambda1, log_lambda2]
xt_res, xt_data, u_data = xt_res.double(), xt_data.double(), u_data.double()

**(2) Training.** The loop of Section 3, with `step(params, state)`: one iteration of the chosen method. A Levenberg–Marquardt iteration is much more expensive than a BFGS iteration, but worth many of them: 50 iterations instead of 4000.

In [ ]:
# ---- Training at second order (starts from the state after Adam)
N_ITER, LOG_EVERY = {"SSBFGS": (4000, 100), "LM": (50, 1)}[METHOD]
hist = {"it": [], "time": [], "loss": [], "data": [], "pde": [], "lambda1": [], "lambda2": []}
t0 = time.time()
fig, axs = plt.subplot_mosaic([["loss", "l1", "e1"], ["loss", "l2", "e2"]], figsize=(17, 6), layout="constrained")
live = display(fig, display_id=True); plt.close(fig)

for it in tqdm(range(N_ITER), desc=METHOD):
    # 1. one iteration of the method
    step(params, state)

    # 2. monitoring (at the current parameters) and live plot
    if it % LOG_EVERY == 0 or it == N_ITER - 1:
        u, u_x, u_t, u_xx = derivatives(model, xt_res)
        r = u_t + lambda1 * u * u_x - torch.exp(log_lambda2) * u_xx
        loss_pde, loss_data = (r ** 2).mean(), ((model(xt_data) - u_data) ** 2).mean()
        hist["it"].append(it + 1)
        hist["time"].append(time.time() - t0)
        hist["loss"].append((W_DATA * loss_data + W_PDE * loss_pde).item())
        hist["data"].append(loss_data.item())
        hist["pde"].append(loss_pde.item())
        hist["lambda1"].append(lambda1.item())
        hist["lambda2"].append(torch.exp(log_lambda2).item())

        l1, l2 = np.array(hist["lambda1"]), np.array(hist["lambda2"])
        for ax in axs.values(): ax.clear()
        axs["loss"].semilogy(hist["it"], hist["loss"], color="tab:blue", lw=2.2, alpha=0.5, label="total loss")
        axs["loss"].semilogy(hist["it"], hist["data"], color="tab:green", label="data")
        axs["loss"].semilogy(hist["it"], hist["pde"], color="tab:orange", label="PDE")
        axs["l1"].plot(hist["it"], l1); axs["l1"].axhline(LAMBDA1_TRUE, color="k", ls="--")
        axs["l2"].plot(hist["it"], l2); axs["l2"].axhline(LAMBDA2_TRUE, color="k", ls="--")
        axs["e1"].semilogy(hist["it"], np.abs(l1 - LAMBDA1_TRUE) / LAMBDA1_TRUE)
        axs["e2"].semilogy(hist["it"], np.abs(l2 - LAMBDA2_TRUE) / LAMBDA2_TRUE)
        axs["loss"].legend(); axs["loss"].set_title(f"{METHOD}: loss = {W_DATA:g} data + {W_PDE:g} PDE")
        axs["l1"].set_title(r"$\lambda_1$ (dashed: true)"); axs["l2"].set_title(r"$\lambda_2$ (dashed: true)")
        axs["e1"].set_title(r"relative error on $\lambda_1$"); axs["e2"].set_title(r"relative error on $\lambda_2$")
        for ax in axs.values(): ax.set_xlabel("iteration")
        live.update(fig)

print(f"after {METHOD}: {N_ITER} iterations, {hist['time'][-1]:.0f} s | loss {hist['loss'][-1]:.2e} "
      f"(data {hist['data'][-1]:.1e}, PDE {hist['pde'][-1]:.1e}) | lambda1 = {lambda1.item():.6f}, lambda2 = {torch.exp(log_lambda2).item():.5e}")

**(3) Check.** As in Section 3.3.

With self-scaled BFGS (4000 iterations, in our tests), the loss goes down to about $10^{-11}$ to $10^{-10}$, about 5 orders of magnitude below L-BFGS: $\lambda_1$ is found with an error of about $10^{-6}$, $\lambda_2$ with an error of a few $10^{-5}$ (it fluctuates between $10^{-5}$ and $3\cdot10^{-4}$ along the training), and $u$ with a relative error of about $2\cdot10^{-5}$. The difficulty with $\lambda_2$ was the optimizer, not the problem.

In [ ]:
l1, l2 = lambda1.item(), torch.exp(log_lambda2).item()
with torch.no_grad():
    U = model(torch.tensor(XT_REF, dtype=torch.float64, device=DEVICE)).cpu().numpy().reshape(U_REF.shape)
print(f"lambda1 = {l1:.6f}    (true {LAMBDA1_TRUE:.6f})    -> error {100 * abs(l1 - LAMBDA1_TRUE) / LAMBDA1_TRUE:.3f} %")
print(f"lambda2 = {l2:.5e} (true {LAMBDA2_TRUE:.5e}) -> error {100 * abs(l2 - LAMBDA2_TRUE) / LAMBDA2_TRUE:.3f} %")
print(f"relative L2 error on u: {np.linalg.norm(U - U_REF) / np.linalg.norm(U_REF):.1e}")

fig, axs = plt.subplots(1, 4, figsize=(17, 3.6))
im = axs[0].imshow(np.abs(U - U_REF), extent=[0, T_MAX, -1, 1], origin="lower", aspect="auto", cmap="magma_r")
fig.colorbar(im, ax=axs[0]); axs[0].set(title=r"$|u_\theta - u|$", xlabel="$t$", ylabel="$x$")
for ax, k in zip(axs[1:], [25, 50, 75]):
    ax.plot(X_REF, U_REF[:, k], "k", lw=3, alpha=0.3, label="exact")
    ax.plot(X_REF, U[:, k], "r--", lw=1.5, label=METHOD)
    ax.set(title=f"$t = {T_REF[k]:.2f}$", xlabel="$x$")
axs[1].legend(); plt.tight_layout(); plt.show()

## 5. Exercise: another pair $(\lambda_1, \lambda_2)$

1. In Section 1, choose another pair, e.g. `LAMBDA1_TRUE, LAMBDA2_TRUE = 0.5, 0.01` (slower advection, thicker front) or `2.0, 0.005` (steeper front). Look at the exact solution in the figure of the data points, then run the notebook again. Are $\lambda_1$ and $\lambda_2$ still found? With which accuracy, compared with the default pair?
2. The initial guess ($\lambda_1 = 0$, $\log\lambda_2 = -6$, i.e. $\lambda_2 \approx 2.5\cdot10^{-3}$) does not change. Does it matter when the true $\lambda_2$ is much larger, e.g. $0.05$?
3. Which pairs are the hardest to identify, and why? Hint: how large is the term $\lambda_2\,\partial_{xx}u$ outside the front, and how many data points lie in the front?
4. Go back to the default pair. With self-scaled BFGS, how does the error on $\lambda_2$ change with `N_DATA` (500, 2000, 5000)? Instead of adding uniform points, add data points only near the front ($|x| < 0.05$, $t > 0.3$): how many are needed to reach the same accuracy?